# PureMagic Tutorial

**PureMagic** is a dynamic lattice-surgery scheduler for quantum surface-code topologies. Given a
Clifford+T circuit, it schedules the circuit's Pauli-product operations onto an abstract physical
layout, routing them with Steiner-tree packing to maximize parallelism, while dynamically
simulating magic-state cultivation and T-gate injection failures.

- Lattice-surgery scheduling: [Game of Surface Codes](https://arxiv.org/abs/1808.02892)
- The paper this repository accompanies: [PureMagic: a Dynamic Scheduler for Lattice Surgery](https://arxiv.org/pdf/2512.06484)
  (to appear at [QCE](https://qce.quantum.ieee.org/); the code/data snapshot for that paper is
  tagged [`QCE`](https://github.com/BQSKit/PureMagic/tree/QCE) in this repository)

This notebook walks through the full pipeline -- QASM in, schedule out -- using the actual
`puremagic`/`transpile`/`compile_cliffordt`/`circuit_stats` binaries built from this repo, plus the
Python analysis tooling in `data_processing/`. It assumes you've already built the project:

```bash
cargo build --release
```

(see the top-level [README](../README.md) for the Git LFS / submodule notes that building depends
on). Everything below drives the release binaries with `subprocess` and inspects their output --
there is no Python API for the scheduler itself, it's a Rust CLI tool.


In [ ]:
%matplotlib inline

import glob
import os
import pathlib
import re
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, SVG, display


def find_repo_root(start="."):
    """Walk upward from `start` looking for the PureMagic repo root."""
    p = pathlib.Path(start).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "Cargo.toml").exists() and (candidate / "data_processing").is_dir():
            return candidate
    raise RuntimeError("Could not locate the PureMagic repo root (looked for Cargo.toml)")


REPO_ROOT = find_repo_root(".")
BIN = REPO_ROOT / "target" / "release"

for exe in ["compile_cliffordt", "transpile", "puremagic", "circuit_stats"]:
    if not (BIN / exe).exists():
        raise RuntimeError(f"{exe} not found under {BIN} -- run `cargo build --release` first")

# All the Python analysis helpers live in data_processing/, as plain importable modules.
sys.path.insert(0, str(REPO_ROOT / "data_processing"))
import plot_puremagic as pm  # noqa: E402  (parse_output_file, etc.)

# Everything this notebook generates (circuits, schedules, plots) goes in a scratch directory,
# not the repo, so re-running the notebook never leaves stray files behind.
work_dir = pathlib.Path(tempfile.mkdtemp(prefix="puremagic_tutorial_"))
print("Repo root:", REPO_ROOT)
print("Scratch working directory:", work_dir)


def render_terminal_output(text):
    """Replay '\\r'/'\\b' cursor movement so captured stdout prints the way it would
    look on a live terminal, instead of dumping every intermediate progress-ticker
    frame one after another (puremagic prints a live 'NN%' scheduling progress
    ticker this way).
    """
    lines = []
    buf = []
    cursor = 0
    for ch in text:
        if ch == "\n":
            lines.append("".join(buf))
            buf, cursor = [], 0
        elif ch == "\r":
            cursor = 0
        elif ch == "\x08":
            cursor = max(0, cursor - 1)
        else:
            if cursor < len(buf):
                buf[cursor] = ch
            else:
                buf.append(ch)
            cursor += 1
    lines.append("".join(buf))
    return "\n".join(lines)


def collapse_repeated_lines(text, pattern, min_repeat=4):
    """Collapse runs of >= min_repeat consecutive lines matching `pattern` into a
    single summary line. `--plot circuit,cstats,paths` prints a 'Plotted topology
    to <name>.N.png' / 'Timing: ... took ... s' pair per routed product -- real
    but repetitive, and not worth scrolling through in full here.
    """
    rx = re.compile(pattern)
    lines = text.split("\n")
    out = []
    i = 0
    while i < len(lines):
        if rx.match(lines[i]):
            j = i
            while j < len(lines) and rx.match(lines[j]):
                j += 1
            if j - i >= min_repeat:
                out.append(f"... ({j - i} similar lines omitted) ...")
                i = j
                continue
        out.append(lines[i])
        i += 1
    return "\n".join(out)


## 1. The pipeline

PureMagic schedules circuits already expressed as a sequence of Pauli-product rotations/measurements
(the `.trans` format). Getting there from an arbitrary QASM circuit is a 3-stage pipeline, each
stage its own Rust binary:

```
circuit.qasm
   |
   |  compile_cliffordt   (any gate set -> Clifford+T)
   v
circuit.cliffordt.qasm
   |
   |  transpile           (Clifford+T -> Pauli products)
   v
circuit.trans
   |
   |  puremagic           (Pauli products -> schedule)
   v
circuit.schedule
```

1. **`compile_cliffordt`** -- compiles an arbitrary circuit down to the Clifford+T gate set (phase-polynomial
   merge, gauge collapse, windowed resynthesis, final per-block synthesis via `cyclosynth`, then a
   Clifford-run simplification pass). Skip this stage if your circuit is already Clifford+T.
2. **`transpile`** -- rewrites the Clifford+T circuit into Pauli-product form: each output line is a signed
   Pauli string tagged with a gate (`<T>`, `<CX>`, `<S>`, `<SX>`, `<M>`, `<Z>`, `<X>`).
3. **`puremagic`** -- the scheduler itself: lays the circuit out on a surface-code topology, routes each
   product with Steiner-tree packing, simulates magic-state cultivation and T-gate injection failures, and
   writes out a logical-cycle-by-logical-cycle schedule.

We'll run all three stages on a small hand-written example circuit, then look at what each stage produced.


## 2. A worked example

A tiny 5-qubit circuit: a chain of `CX` gates for entanglement, four `rz(pi/4)` rotations sharing one
angle, and one `rz` at an arbitrary angle that shares nothing with the others.

In [ ]:
example_qasm = work_dir / "example.qasm"
example_qasm.write_text("""\
OPENQASM 2.0;
include "qelib1.inc";
qreg q[5];
h q[0];
cx q[0],q[1];
rz(0.7853981633974483) q[1];
cx q[1],q[2];
rz(0.7853981633974483) q[2];
cx q[2],q[3];
rz(1.9106) q[3];
cx q[3],q[4];
rz(0.7853981633974483) q[4];
h q[4];
cx q[0],q[4];
rz(2.356194490192345) q[0];
""")

print(example_qasm.read_text())


### Step 1 -- compile to Clifford+T

`compile_cliffordt` writes `<name>.cliffordt.qasm` next to the input by default. Watch the per-stage
summary it prints: stage 0/1 recognize that four of our five `rz` rotations are exact multiples of
pi/4 -- dyadic angles it can express exactly in Clifford+T with no numerical synthesis at all -- and
fold them away for free ("net -4" Rz remaining after gauge collapse). That leaves only the one
genuinely arbitrary angle (`rz(1.9106)`) for stage 3's numerical synthesis to turn into real T gates,
which is why the gate count jumps from a dozen gates to a couple hundred. On a circuit with no
repeated or dyadic angles at all, *every* rotation would take that same expensive path (see
[`data/circuits/README.md`](../data/circuits/README.md) for how differently the eleven benchmark
circuits stress this stage).


In [ ]:
def run(cmd, **kwargs):
    """Run a command in work_dir, printing it and its combined output; raise on failure."""
    printable = " ".join(str(c) for c in cmd)
    print(f"$ {printable}")
    result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True, **kwargs)
    cleaned = render_terminal_output(result.stdout)
    cleaned = collapse_repeated_lines(cleaned, r"^(Plotted topology to .*\.png|Timing: .*took .* s)$")
    print(cleaned)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"command failed ({result.returncode}): {printable}")
    return result


run([BIN / "compile_cliffordt", "example.qasm"])


### Step 2 -- transpile to `.trans` format

Each output line is a signed Pauli product tagged with the gate it implements:

| Tag | Meaning |
|-----|---------|
| `<T>` | T gate (pi/8 rotation) |
| `<CX>` | CX Clifford gate |
| `<S>` | S/Sdg Clifford gate |
| `<SX>` | SX/SXdg Clifford gate |
| `<M>` | Measurement |
| `<Z>` | Z Pauli gate |
| `<X>` | X Pauli gate |

and each qubit's column is one of `_` (identity), `X`, `Y`, `Z`. `--defer_trailing` holds back each
qubit's trailing single-qubit Clifford run at a flush instead of always emitting it immediately, which
reduces logical cycles at any weight -- worth turning on by default.


In [ ]:
run([BIN / "transpile", "-i", "example.cliffordt.qasm", "--defer_trailing"])

example_trans = work_dir / "example.trans0"
trans_lines = example_trans.read_text().splitlines()
print(f"{len(trans_lines)} lines total; first 8 and last 5:\n")
print("\n".join(trans_lines[:8]))
print("...")
print("\n".join(trans_lines[-5:]))


Reading the file: `+XZ___<T>` is a T gate applying a Pauli-product rotation across qubits 0 and 1
(`X` on qubit 0, `Z` on qubit 1); the long run of near-identical weight-4 products in the middle is
exactly what you'd expect from synthesizing that one arbitrary-angle rotation from Step 1 -- a single
logical rotation becomes a long sequence of T gates acting on (almost) the same support, one for each
T gate in its numerical decomposition. The final block of `<M>` lines are the measurements that close
out the circuit. Notice there are no Clifford (`<CX>`/`<S>`/`<SX>`) lines left at all here --
`transpile` folded every Clifford gate into the Pauli frame of the surrounding products, which is why
the tool reports "Clifford reduction: ... 100%" above.


### Step 3 -- schedule with `puremagic`

This is the main event. `--plot topo,circuit,cstats,paths` asks for every available visualization; in
practice you'd pick only the ones you need, since `paths` in particular writes one PNG per logical
cycle (capped at the first 100).


In [ ]:
run([BIN / "puremagic", "--circuit", "example.trans0", "--plot", "topo,circuit,cstats,paths"])


### Reading the scheduler's report

A few lines from that output are worth calling out:

- **Number of qubits (data/bus/magic)** -- how the auto-generated topology's qubits are split. With no
  `--topo` given, PureMagic sizes a grid from the circuit's qubit count and `--ancilla-rows`.
- **Magic state cultivation time** -- cultivation cycles are drawn from an exponential distribution
  parameterized by `--magic-state-lambda`; `average`/`min`/`max` here are over every cultivation attempt
  actually needed during scheduling.
- **T gate failures** -- T-gate injection can fail and need a retry (unless `--no-t-failures` is set);
  this reports how many of the circuit's T gates failed at least once.
- **Parallelism** / **Normalized scheduling efficiency** -- parallelism is (products scheduled) /
  (logical cycles used); scheduling efficiency compares the actual logical-cycle count to an optimal
  lower bound computed for the same circuit.
- **Volume** -- logical cycles x total qubits; the headline cost metric.

And the output files it just wrote, in `work_dir`:

| File | Contents |
|------|----------|
| `example.schedule` | Final schedule (lcycle -> operations) |
| `example.topo.png` / `example.topo.txt` | Topology visualization / grid dump |
| `example.circuit/` | Circuit layer plots (one PNG) |
| `example.layer_stats.svg` | Circuit layer statistics over time |
| `example.paths/` | Per-lcycle routing-path visualizations (first 100 lcycles) |


In [ ]:
display(Image(filename=str(work_dir / "example.topo.png")))


The auto-generated topology: `b` bus qubits (routing), `m` magic-state qubits, and `d` data qubits
(each data qubit is really a pair of grid cells, marked `X`/`Z` for its two stabilizer boundaries). The
same layout is written as plain text to `<name>.topo.txt` -- this is exactly the format expected by
`puremagic --topo <file>` if you want to hand-design (or reuse) a topology instead of auto-generating one:


In [ ]:
print((work_dir / "example.topo.txt").read_text())


In [ ]:
path_pngs = sorted((work_dir / "example.paths").glob("*.png"), key=lambda p: int(p.stem.split(".")[-1]))
display(Image(filename=str(path_pngs[0])))


Each `paths/` image is a snapshot of one logical cycle: the Steiner tree(s) routed that cycle
overlaid on the topology. A boxed `X`/`Z` label marks where a routed Pauli product touches a data
qubit's X or Z boundary, colored to match that product's own route color. Magic-state qubits show
either a countdown of remaining cultivation cycles (a plain number) or a bold `T` once a state has
finished cultivating and is ready to be consumed. Note that T-gate injection success/failure itself
isn't drawn per-cycle at all -- it's only ever reported in aggregate, as the `T gate failures: N/M`
line in the scheduler's text summary. The caption reports how many products were scheduled that cycle
relative to how many were available, and what fraction of qubits were in use.


In [ ]:
display(SVG(filename=str(work_dir / "example.layer_stats.svg")))


`layer_stats.svg` tracks the circuit's own DAG structure over layers -- average/max products per layer
and average/max product weight -- independent of scheduling; it's a property of the transpiled circuit,
not of any particular schedule.

Finally, the schedule itself: one line per operation, `<lcycle>: <Pauli string> <product id><gate tag>`.


In [ ]:
schedule_lines = (work_dir / "example.schedule").read_text().splitlines()
print("\n".join(schedule_lines[:10]))


## 3. Parameters that shape the schedule

The scheduler's behavior is controlled by a handful of flags (see `puremagic -h` for the full list):

| Flag | Default | Effect |
|------|---------|--------|
| `-m, --magic-state-lambda` | `0.0387396` | Magic states produced per magic qubit per lcycle -- higher means faster cultivation, more magic-state throughput |
| `-u, --use-magic-routing` | off | Lets magic qubits double as routing qubits, not just bus qubits |
| `-a, --ancilla-rows` | `1` | Rows of ancilla between data patches (magic routing only) |
| `-S, --sides_only` | off | Route only through the sides of data patches, not top/bottom |
| `-F, --no-t-failures` | off | Disable T-gate injection failures entirely |
| `-R, --randomize-data-qubits` | off | Randomize the data-qubit numbering on the topology |
| `-r, --rseed` | `29` | Random seed (cultivation times and T-failure draws) |

`--magic-state-lambda` is the one with the most direct effect on how magic-state-bound a schedule is, so
let's sweep it and see.


### Parameter sweep: `magic_state_lambda`

We re-run the same circuit at several lambda values, capturing each run's full stdout to a log file --
that's the format `data_processing/plot_puremagic.py`'s `parse_output_file` expects (it's built to parse
`puremagic`'s own stdout, the same logs `plot_all.sh` collects when driving comparisons across the full
benchmark suite).


In [ ]:
sweep_dir = work_dir / "sweep"
sweep_dir.mkdir()

lambdas = [0.02, 0.0387396, 0.08, 0.15, 0.30]
log_paths = []
for lam in lambdas:
    result = run([BIN / "puremagic", "--circuit", "example.trans0", "-m", str(lam)])
    log_path = sweep_dir / f"lambda_{lam}.out"
    log_path.write_text(result.stdout)
    log_paths.append(log_path)


In [ ]:
df = pd.concat([pm.parse_output_file(str(p)) for p in log_paths], ignore_index=True)
df[["magic_state_lambda", "scheduling_efficiency", "lcycles", "volume"]]


Raising lambda (faster cultivation, i.e. more magic-state throughput) shortens the schedule -- but only
up to a point: once magic states stop being the bottleneck, the circuit's own dependency structure takes
over and further increases in lambda stop helping at all. That crossover is exactly the kind of tradeoff
`plot_all.sh`'s sweeps are built to characterize across the full benchmark suite.


In [ ]:
fig, (ax_lcycles, ax_volume) = plt.subplots(1, 2, figsize=(10, 4))

# Reuse plot_puremagic.py's own categorical color, for consistency with the rest of this repo's plots.
colour = pm._COLOURS[0]

ax_lcycles.plot(df["magic_state_lambda"], df["lcycles"], marker="o", color=colour)
ax_lcycles.set_xlabel("magic_state_lambda")
ax_lcycles.set_ylabel(pm._Y_AXES["lcycles"])
ax_lcycles.set_title("Schedule length vs. magic-state throughput")

ax_volume.plot(df["magic_state_lambda"], df["volume"], marker="o", color=colour)
ax_volume.set_xlabel("magic_state_lambda")
ax_volume.set_ylabel(pm._Y_AXES["volume"])
ax_volume.set_title("Volume vs. magic-state throughput")

fig.tight_layout()
plt.show()


## 4. Quick estimates without full scheduling: `circuit_stats`

Running the full scheduler is fast, but for a quick sanity check of a circuit's expected layer/volume
range -- before committing to a full scheduling run -- `circuit_stats` computes the same layer and volume
*estimates* directly from the circuit's DAG structure, with no simulation at all.


In [ ]:
run([BIN / "circuit_stats", "--circuit", "example.trans0"])


## 5. Real benchmark circuits

`data/circuits/` holds a curated eleven-circuit suite -- spanning chemistry/physics simulation,
optimization, machine learning, reversible arithmetic, and a structured transform -- chosen to stress
different parts of the compiler and scheduler in different ways (angle-reuse patterns, gate-set
structure, dependency depth, and so on); see [`data/circuits/README.md`](../data/circuits/README.md)
for what makes each one interesting. It's a broader stress-test set than any single paper run would
use, not a fixed "the paper's benchmarks" list. Those circuits are much larger than our toy example --
but the scheduler is fast enough that a full run still only takes a fraction of a second.

Only each circuit's `.qasm` and `.cliffordt.qasm` are checked into the repo (via Git LFS -- run
`git lfs pull` first if `dnn_n51.cliffordt.qasm` below checks out as a small pointer stub rather than
real content); the transpiled `.trans*` files are scratch output (gitignored, regenerated on demand),
so we run `transpile` ourselves here exactly as in Step 2, just on a bigger circuit.


In [ ]:
benchmark_source = REPO_ROOT / "data" / "circuits" / "dnn_n51.cliffordt.qasm"
run([BIN / "transpile", "-i", str(benchmark_source), "--defer_trailing"])
benchmark_circuit = work_dir / "dnn_n51.trans0"

result = subprocess.run(
    [BIN / "circuit_stats", "--circuit", benchmark_circuit],
    capture_output=True, text=True,
)
print(render_terminal_output(result.stdout))


In [ ]:
result = subprocess.run(
    [BIN / "puremagic", "--circuit", benchmark_circuit],
    capture_output=True, text=True,
    cwd=work_dir,  # benchmark_circuit is already inside the scratch dir; schedule lands there too
)
print(render_terminal_output(result.stdout)[-1200:])


## 6. Analyzing results at scale

Everything above works one run at a time; `data_processing/` is where the analysis scales up. A few
scripts worth knowing about (all standalone, run with `-h` for options -- see
[`data_processing/README.md`](../data_processing/README.md) for the complete list):

- **`plot_puremagic.py`** -- the unified results plotter used above via `parse_output_file`; it's also a
  CLI in its own right, comparing multiple `-f file[,file2]:label` series against any x/y-axis pair, with
  ratio-mode (`file1,file2`) for A/B comparisons like PureMagic vs. bus routing. `plot_all.sh` drives it
  across a standard set of comparison plots over the benchmark suite.
- **`scheduling_table.py`** / **`circuit_table.py`** -- generate LaTeX tables of scheduling/circuit
  statistics straight from `puremagic` stdout logs.
- **`mcmc_cultivation.py`** -- MCMC simulation of the magic-state cultivation time distribution.
- **`flasq_lower_bound.py`** / **`dascot_qubit_count.py`** -- independent lower-bound / qubit-count
  estimators used as points of comparison against PureMagic's own numbers.

As a concrete example, here's `plot_puremagic.py` run as a CLI over the lambda sweep from section 3 --
the same `-x cultivation -y lcycles` pairing we plotted by hand above, but produced by the same
plotting tool this repo's larger-scale comparisons are built on:


In [ ]:
circuits_file = work_dir / "circuits.txt"
circuits_file.write_text("example\n")

# plot_puremagic.py expects one log per (x, y) point in a single file; concatenate the sweep logs.
combined_log = sweep_dir / "combined.out"
combined_log.write_text("\n".join(p.read_text() for p in log_paths))

sweep_plot = work_dir / "lambda_sweep.png"
run([
    sys.executable, str(REPO_ROOT / "data_processing" / "plot_puremagic.py"),
    "-x", "cultivation", "-y", "lcycles",
    "-c", str(circuits_file),
    "--lines-with-markers", "--no-legend",
    "-f", str(combined_log),
    "-o", str(sweep_plot),
])
display(Image(filename=str(sweep_plot)))


## Further reading

- Top-level [README](../README.md) -- full CLI reference, circuit/topology file formats, output files.
- [`data_processing/README.md`](../data_processing/README.md) -- every analysis script in this repo.
- [`data/circuits/README.md`](../data/circuits/README.md) -- the eleven-circuit benchmark suite in detail.
- The two papers linked at the top of this notebook -- Game of Surface Codes, and PureMagic's own paper
  -- for the algorithms this implementation is built on.

From here, a natural next step is pointing this notebook's pipeline at one of your own circuits, or
re-running section 3's sweep over a different flag (`--use-magic-routing`, `--sides_only`,
`--no-t-failures`) to see how it trades off against `magic_state_lambda`.
